# Άσκηση 1 — Βάσεις Δεδομένων
### Τσαγκανός Αλέξανδρος

**Weak-entity υλοποίηση** (τελική): το `ΜΑΘΗΜΑ` είναι ασθενής οντότητα — το διπλό ορθογώνιο και το διακεκομμένα υπογραμμισμένο `ΚωδικόςΜ` στο ΔΟΣ το επιβεβαιώνουν. `PK(ΜΑΘΗΜΑ)=(ΚωδικόςΤ,ΚωδικόςΜ)`, με propagation σε `ΠΡΟΤΕΙΝΕΙ` και `ΔΗΛΩΝΕΙ`.

Τρέχει end-to-end σε SQLite (Python built-in). Χρήση:  
- Κελί 1: setup & schema  
- Κελί 2: εισαγωγή δεδομένων + FK check  
- Κελιά 3–12: ερωτήματα Q1–Q10

Η συνάρτηση `q(sql)` εκτελεί και επιστρέφει `pandas.DataFrame`.

In [1]:
import sqlite3, pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

def q(sql):
    return pd.read_sql_query(sql, conn)

def run(sql):
    cur.executescript(sql); conn.commit()

schema = r'''
DROP TABLE IF EXISTS ΔΗΛΩΝΕΙ;
DROP TABLE IF EXISTS ΠΡΟΤΕΙΝΕΙ;
DROP TABLE IF EXISTS ΣΥΓΓΡΑΜΜΑ;
DROP TABLE IF EXISTS ΦΟΙΤΗΤΗΣ;
DROP TABLE IF EXISTS ΜΑΘΗΜΑ;
DROP TABLE IF EXISTS ΤΜΗΜΑ;
DROP TABLE IF EXISTS ΠΑΝΕΠΙΣΤΗΜΙΟ;

CREATE TABLE ΠΑΝΕΠΙΣΤΗΜΙΟ (
    ΚωδικόςΠ  INTEGER PRIMARY KEY,
    Ονομασία  TEXT NOT NULL
);

CREATE TABLE ΤΜΗΜΑ (
    ΚωδικόςΤ  INTEGER PRIMARY KEY,
    Ονομασία  TEXT NOT NULL,
    ΚωδικόςΠ  INTEGER NOT NULL,
    FOREIGN KEY (ΚωδικόςΠ) REFERENCES ΠΑΝΕΠΙΣΤΗΜΙΟ(ΚωδικόςΠ)
);

-- ΜΑΘΗΜΑ: ασθενής, PK = (ΚωδικόςΤ, ΚωδικόςΜ)
CREATE TABLE ΜΑΘΗΜΑ (
    ΚωδικόςΤ        INTEGER NOT NULL,
    ΚωδικόςΜ        INTEGER NOT NULL,
    Τίτλος          TEXT NOT NULL,
    ΈτοςΣπουδών     INTEGER CHECK (ΈτοςΣπουδών BETWEEN 1 AND 6),
    ΕξάμηνοΣπουδών  INTEGER CHECK (ΕξάμηνοΣπουδών BETWEEN 1 AND 12),
    PRIMARY KEY (ΚωδικόςΤ, ΚωδικόςΜ),
    FOREIGN KEY (ΚωδικόςΤ) REFERENCES ΤΜΗΜΑ(ΚωδικόςΤ) ON DELETE CASCADE
);

CREATE TABLE ΦΟΙΤΗΤΗΣ (
    ΚωδικόςΦ      INTEGER PRIMARY KEY,
    ΈτοςΕγγραφής  INTEGER NOT NULL,
    ΚωδικόςΤ      INTEGER NOT NULL,
    UNIQUE (ΚωδικόςΦ, ΚωδικόςΤ),
    FOREIGN KEY (ΚωδικόςΤ) REFERENCES ΤΜΗΜΑ(ΚωδικόςΤ)
);

CREATE TABLE ΣΥΓΓΡΑΜΜΑ (
    ΚωδικόςΣ        INTEGER PRIMARY KEY,
    Τίτλος          TEXT NOT NULL,
    Συγγραφείς      TEXT,
    ΑριθμόςΈκδοσης  INTEGER CHECK (ΑριθμόςΈκδοσης > 0),
    ΈτοςΈκδοσης     INTEGER,
    ΕκδοτικόςΟίκος  TEXT,
    Τιμή            REAL CHECK (Τιμή IS NULL OR Τιμή >= 0),
    URL             TEXT
);

CREATE TABLE ΠΡΟΤΕΙΝΕΙ (
    ΚωδικόςΤ  INTEGER NOT NULL,
    ΚωδικόςΜ  INTEGER NOT NULL,
    ΚωδικόςΣ  INTEGER NOT NULL,
    Σειρά     INTEGER,
    PRIMARY KEY (ΚωδικόςΤ, ΚωδικόςΜ, ΚωδικόςΣ),
    UNIQUE      (ΚωδικόςΤ, ΚωδικόςΜ, Σειρά),
    FOREIGN KEY (ΚωδικόςΤ, ΚωδικόςΜ) REFERENCES ΜΑΘΗΜΑ(ΚωδικόςΤ, ΚωδικόςΜ)
        ON DELETE CASCADE,
    FOREIGN KEY (ΚωδικόςΣ) REFERENCES ΣΥΓΓΡΑΜΜΑ(ΚωδικόςΣ)
);

CREATE TABLE ΔΗΛΩΝΕΙ (
    ΚωδικόςΦ  INTEGER NOT NULL,
    ΚωδικόςΤ  INTEGER NOT NULL,
    ΚωδικόςΜ  INTEGER NOT NULL,
    ΚωδικόςΣ  INTEGER NOT NULL,
    PRIMARY KEY (ΚωδικόςΦ, ΚωδικόςΤ, ΚωδικόςΜ),
    FOREIGN KEY (ΚωδικόςΦ, ΚωδικόςΤ)
        REFERENCES ΦΟΙΤΗΤΗΣ(ΚωδικόςΦ, ΚωδικόςΤ),
    FOREIGN KEY (ΚωδικόςΤ, ΚωδικόςΜ, ΚωδικόςΣ)
        REFERENCES ΠΡΟΤΕΙΝΕΙ(ΚωδικόςΤ, ΚωδικόςΜ, ΚωδικόςΣ)
);
'''
run(schema)
print("Schema created.")

Schema created.


In [2]:
data = r'''
INSERT INTO ΠΑΝΕΠΙΣΤΗΜΙΟ VALUES
 (1,'ΠΑΝΕΠΙΣΤΗΜΙΟ ΠΑΤΡΩΝ'),(2,'ΕΚΠΑ'),(3,'ΑΠΘ');

INSERT INTO ΤΜΗΜΑ VALUES
 (10,'ΜΑΘΗΜΑΤΙΚΟ',1),(11,'ΠΛΗΡΟΦΟΡΙΚΗΣ',1),
 (20,'ΜΑΘΗΜΑΤΙΚΟ',2),(30,'ΦΥΣΙΚΗΣ',3);

-- ΚωδικόςΜ=1 επαναλαμβάνεται σε 3 τμήματα: νόμιμο λόγω composite PK
INSERT INTO ΜΑΘΗΜΑ(ΚωδικόςΤ,ΚωδικόςΜ,Τίτλος,ΈτοςΣπουδών,ΕξάμηνοΣπουδών) VALUES
 (10,1,'Άλγεβρα Ι',1,1),
 (10,2,'Ανάλυση Ι',1,1),
 (11,3,'Βάσεις Δεδομένων',3,5),
 (11,4,'Αλγόριθμοι',2,3),
 (20,1,'Γραμμική Άλγεβρα',1,1),
 (30,1,'Μηχανική',1,2);

INSERT INTO ΣΥΓΓΡΑΜΜΑ VALUES
 (501,'Άλγεβρα','Αρτεμιάδης',3,2010,'ΤΖΙΟΛΑ',45.00,'http://x/501'),
 (502,'Εισαγωγή στη Μαθηματική Ανάλυση','Ρασσιάς',2,2008,'ΠΕΚ',NULL,NULL),
 (503,'Θεμελιώδεις Δομές','Aho',1,1998,'ΤΖΙΟΛΑ',60.00,'http://x/503'),
 (504,'Συστήματα ΒΔ','Elmasri',7,2016,'ΤΖΙΟΛΑ',55.00,'http://x/504'),
 (505,'Αλγόριθμοι','CLRS',3,2009,'ΚΛΕΙΔΑΡΙΘΜΟΣ',80.00,'http://x/505'),
 (506,'Διακριτά Μαθηματικά','Rosen',7,2012,'ΠΕΚ',50.00,NULL),
 (507,'Γραμμική Άλγεβρα','Strang',4,2005,'ΚΛΕΙΔΑΡΙΘΜΟΣ',NULL,NULL),
 (508,'Μηχανική','Halliday',10,2018,'ΓΚΙΟΥΡΔΑΣ',70.00,'http://x/508');

INSERT INTO ΠΡΟΤΕΙΝΕΙ(ΚωδικόςΤ,ΚωδικόςΜ,ΚωδικόςΣ,Σειρά) VALUES
 (10,1,501,1),(10,1,506,2),
 (10,2,502,1),
 (11,3,504,1),
 (11,4,505,1),(11,4,503,2),
 (20,1,507,1),
 (30,1,508,1);

INSERT INTO ΦΟΙΤΗΤΗΣ(ΚωδικόςΦ,ΈτοςΕγγραφής,ΚωδικόςΤ) VALUES
 (1001,2022,10),(1002,2023,10),
 (1003,2021,11),(1004,2022,11),(1005,2023,11),
 (2001,2022,20),(3001,2023,30);

INSERT INTO ΔΗΛΩΝΕΙ(ΚωδικόςΦ,ΚωδικόςΤ,ΚωδικόςΜ,ΚωδικόςΣ) VALUES
 (1001,10,1,501),(1001,10,2,502),(1002,10,1,506),
 (1003,11,3,504),(1003,11,4,505),
 (1004,11,3,504),(1004,11,4,503),
 (1005,11,3,504),(1005,11,4,505),
 (2001,20,1,507);
'''
run(data)

# FK integrity check
violations = cur.execute("PRAGMA foreign_key_check;").fetchall()
print("FK violations:", violations if violations else "none ✓")
print("Data inserted.")

FK violations: none ✓
Data inserted.


### Q1 — Συγγράμματα χωρίς δηλωμένη τιμή

In [3]:
q('''
SELECT ΚωδικόςΣ, Τίτλος, Συγγραφείς, ΑριθμόςΈκδοσης, ΈτοςΈκδοσης, ΕκδοτικόςΟίκος
FROM ΣΥΓΓΡΑΜΜΑ
WHERE Τιμή IS NULL;
''')

,ΚωδικόςΣ,Τίτλος,Συγγραφείς,ΑριθμόςΈκδοσης,ΈτοςΈκδοσης,ΕκδοτικόςΟίκος
0,502,Εισαγωγή στη Μαθηματική Ανάλυση,Ρασσιάς,2,2008,ΠΕΚ
1,507,Γραμμική Άλγεβρα,Strang,4,2005,ΚΛΕΙΔΑΡΙΘΜΟΣ


### Q2 — ΜΑΘΗΜΑΤΙΚΟ / ΠΑΝΕΠΙΣΤΗΜΙΟ ΠΑΤΡΩΝ

In [4]:
q('''
SELECT Φ.ΚωδικόςΦ, Φ.ΈτοςΕγγραφής
FROM ΦΟΙΤΗΤΗΣ Φ
JOIN ΤΜΗΜΑ        Τ ON Φ.ΚωδικόςΤ = Τ.ΚωδικόςΤ
JOIN ΠΑΝΕΠΙΣΤΗΜΙΟ Π ON Τ.ΚωδικόςΠ = Π.ΚωδικόςΠ
WHERE TRIM(Τ.Ονομασία) = 'ΜΑΘΗΜΑΤΙΚΟ'
  AND TRIM(Π.Ονομασία) = 'ΠΑΝΕΠΙΣΤΗΜΙΟ ΠΑΤΡΩΝ';
''')

,ΚωδικόςΦ,ΈτοςΕγγραφής
0,1001,2022
1,1002,2023


### Q3 — Πλήθος φοιτητών με τουλάχιστον ένα σύγγραμα στην δήλωση

In [5]:
q('''
SELECT COUNT(DISTINCT ΚωδικόςΦ) AS ΠλήθοςΦοιτητώνΜεΤουλάχιστονΈναΣύγγραμμα
FROM ΔΗΛΩΝΕΙ;
''')

,ΠλήθοςΦοιτητώνΜεΤουλάχιστονΈναΣύγγραμμα
0,6


### Q4 — Αντίτυπα ανά εκδότη (αλφαβητικά)

In [6]:
q('''
SELECT Σ.ΕκδοτικόςΟίκος, COUNT(*) AS ΠλήθοςΑντιτύπωνΣυγγραμμάτων
FROM ΔΗΛΩΝΕΙ Δ
JOIN ΣΥΓΓΡΑΜΜΑ Σ ON Δ.ΚωδικόςΣ = Σ.ΚωδικόςΣ
GROUP BY Σ.ΕκδοτικόςΟίκος
ORDER BY Σ.ΕκδοτικόςΟίκος ASC;
''')

,ΕκδοτικόςΟίκος,ΠλήθοςΑντιτύπωνΣυγγραμμάτων
0,ΚΛΕΙΔΑΡΙΘΜΟΣ,3
1,ΠΕΚ,2
2,ΤΖΙΟΛΑ,5


### Q5 — MIN/MAX τιμής ανά εκδότη, >1 συγγράμματα

In [7]:
q('''
SELECT ΕκδοτικόςΟίκος,
       MIN(Τιμή) AS ΕλάχιστηΤιμήΣυγγράμματος,
       MAX(Τιμή) AS ΜέγιστηΤιμήΣυγγράμματος
FROM ΣΥΓΓΡΑΜΜΑ
WHERE ΕκδοτικόςΟίκος IS NOT NULL
GROUP BY ΕκδοτικόςΟίκος
HAVING COUNT(*) > 1;
''')

,ΕκδοτικόςΟίκος,ΕλάχιστηΤιμήΣυγγράμματος,ΜέγιστηΤιμήΣυγγράμματος
0,ΚΛΕΙΔΑΡΙΘΜΟΣ,80.0,80.0
1,ΠΕΚ,50.0,50.0
2,ΤΖΙΟΛΑ,45.0,60.0


### Q6 — Συγγράμματα δηλωμένα από >2 φοιτητές

> Add blockquote



In [8]:
q('''
SELECT ΚωδικόςΣ, COUNT(DISTINCT ΚωδικόςΦ) AS ΠλήθοςΦοιτητών
FROM ΔΗΛΩΝΕΙ
GROUP BY ΚωδικόςΣ
HAVING COUNT(DISTINCT ΚωδικόςΦ) > 2
ORDER BY ΠλήθοςΦοιτητών DESC;
''')

,ΚωδικόςΣ,ΠλήθοςΦοιτητών
0,504,3


### Q7 — Παλαιότερο σύγγραμμα

In [9]:
q('''
SELECT Τίτλος, Συγγραφείς, ΑριθμόςΈκδοσης, ΈτοςΈκδοσης, ΕκδοτικόςΟίκος
FROM ΣΥΓΓΡΑΜΜΑ
WHERE ΈτοςΈκδοσης = (SELECT MIN(ΈτοςΈκδοσης) FROM ΣΥΓΓΡΑΜΜΑ);
''')

,Τίτλος,Συγγραφείς,ΑριθμόςΈκδοσης,ΈτοςΈκδοσης,ΕκδοτικόςΟίκος
0,Θεμελιώδεις Δομές,Aho,1,1998,ΤΖΙΟΛΑ


### Q8 — Μαθήματα με ακριβώς ένα προτεινόμενο βιβλίο (composite grouping!)

In [10]:
q('''
SELECT DISTINCT Δ.ΚωδικόςΦ, Μ.ΚωδικόςΤ, Μ.ΚωδικόςΜ, Μ.Τίτλος AS ΤίτλοςΜαθήματος
FROM ΔΗΛΩΝΕΙ Δ
JOIN ΜΑΘΗΜΑ  Μ
  ON Δ.ΚωδικόςΤ = Μ.ΚωδικόςΤ AND Δ.ΚωδικόςΜ = Μ.ΚωδικόςΜ
WHERE (Μ.ΚωδικόςΤ, Μ.ΚωδικόςΜ) IN (
    SELECT ΚωδικόςΤ, ΚωδικόςΜ FROM ΠΡΟΤΕΙΝΕΙ
    GROUP BY ΚωδικόςΤ, ΚωδικόςΜ
    HAVING COUNT(*) = 1
)
ORDER BY Μ.ΚωδικόςΤ ASC, Μ.ΚωδικόςΜ ASC;
''')

,ΚωδικόςΦ,ΚωδικόςΤ,ΚωδικόςΜ,ΤίτλοςΜαθήματος
0,1001,10,2,Ανάλυση Ι
1,1003,11,3,Βάσεις Δεδομένων
2,1004,11,3,Βάσεις Δεδομένων
3,1005,11,3,Βάσεις Δεδομένων
4,2001,20,1,Γραμμική Άλγεβρα


### Q9 Version A — Εκδότες με κανένα βιβλίο στην Πάτρα (NOT IN)

In [11]:
q('''
SELECT DISTINCT ΕκδοτικόςΟίκος
FROM ΣΥΓΓΡΑΜΜΑ
WHERE ΕκδοτικόςΟίκος IS NOT NULL
  AND ΕκδοτικόςΟίκος NOT IN (
      SELECT Σ2.ΕκδοτικόςΟίκος
      FROM ΣΥΓΓΡΑΜΜΑ Σ2
      JOIN ΠΡΟΤΕΙΝΕΙ Π    ON Π.ΚωδικόςΣ = Σ2.ΚωδικόςΣ
      JOIN ΜΑΘΗΜΑ    Μ    ON Μ.ΚωδικόςΤ = Π.ΚωδικόςΤ
                         AND Μ.ΚωδικόςΜ = Π.ΚωδικόςΜ
      JOIN ΤΜΗΜΑ     Τ    ON Τ.ΚωδικόςΤ = Μ.ΚωδικόςΤ
      JOIN ΠΑΝΕΠΙΣΤΗΜΙΟ U ON U.ΚωδικόςΠ = Τ.ΚωδικόςΠ
      WHERE U.Ονομασία = 'ΠΑΝΕΠΙΣΤΗΜΙΟ ΠΑΤΡΩΝ'
        AND Σ2.ΕκδοτικόςΟίκος IS NOT NULL
  )
ORDER BY ΕκδοτικόςΟίκος ASC;
''')

,ΕκδοτικόςΟίκος
0,ΓΚΙΟΥΡΔΑΣ


### Q9 Version B — NOT EXISTS per-publisher

In [12]:
q('''
SELECT DISTINCT Σ.ΕκδοτικόςΟίκος
FROM ΣΥΓΓΡΑΜΜΑ Σ
WHERE Σ.ΕκδοτικόςΟίκος IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM ΣΥΓΓΡΑΜΜΑ Σ2
      JOIN ΠΡΟΤΕΙΝΕΙ Π    ON Π.ΚωδικόςΣ = Σ2.ΚωδικόςΣ
      JOIN ΜΑΘΗΜΑ    Μ    ON Μ.ΚωδικόςΤ = Π.ΚωδικόςΤ
                         AND Μ.ΚωδικόςΜ = Π.ΚωδικόςΜ
      JOIN ΤΜΗΜΑ     Τ    ON Τ.ΚωδικόςΤ = Μ.ΚωδικόςΤ
      JOIN ΠΑΝΕΠΙΣΤΗΜΙΟ U ON U.ΚωδικόςΠ = Τ.ΚωδικόςΠ
      WHERE Σ2.ΕκδοτικόςΟίκος = Σ.ΕκδοτικόςΟίκος
        AND U.Ονομασία = 'ΠΑΝΕΠΙΣΤΗΜΙΟ ΠΑΤΡΩΝ'
  )
ORDER BY Σ.ΕκδοτικόςΟίκος ASC;
''')

,ΕκδοτικόςΟίκος
0,ΓΚΙΟΥΡΔΑΣ


### Q10 — Φοιτητές με τις περισσότερες δηλώσεις

In [13]:
q('''
SELECT ΚωδικόςΦ, COUNT(*) AS ΠλήθοςΣυγγραμμάτων
FROM ΔΗΛΩΝΕΙ
GROUP BY ΚωδικόςΦ
HAVING COUNT(*) = (
    SELECT MAX(cnt) FROM (
        SELECT COUNT(*) AS cnt FROM ΔΗΛΩΝΕΙ GROUP BY ΚωδικόςΦ
    ) t
);
''')

,ΚωδικόςΦ,ΠλήθοςΣυγγραμμάτων
0,1001,2
1,1003,2
2,1004,2
3,1005,2


---
### Σημεία ελέγχου

1. `PRAGMA foreign_key_check;` = κενό → weak-entity propagation OK.
2. Q8 χρησιμοποιεί `GROUP BY ΚωδικόςΤ, ΚωδικόςΜ` — κρίσιμο.
3. Q9: δύο ισοδύναμες εκδοχές, ίδιο αποτέλεσμα = `{ΓΚΙΟΥΡΔΑΣ}`.
4. Q10: portable `MAX(subquery)`· εναλλακτικά `RANK() OVER` ή `FETCH FIRST 1 ROW WITH TIES`.
5. Ο φοιτητής 3001 (ΑΠΘ/Φυσικής) δεν έχει καμία δήλωση → Q3=6, Q10 τον εξαιρεί.